# Pairwise Experiments

### Setup

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
import os
load_dotenv()

True

### Task

Let's set up a new task! Here, we have a salesperson named Bob. Bob has a lot of deals, so he wants to summarize what happened in this deals based off of some meeting transcripts.

Bob is iterating on a few different prompts, that will give him nice, concise transcripts for his deals.

Bob has curated a dataset of his deal transcripts, let's go ahead and load that in. You can take a look at the dataset as well if you're curious! Note that this is not a golden dataset, there is no reference output here.

In [2]:
from langsmith import Client

client = Client()
dataset = client.clone_public_dataset(
    "https://smith.langchain.com/public/9078d2f1-7bef-4ba7-b795-210a17682ef9/d"
)
print(f"Dataset cloned successfully: '{dataset.name}' (ID: {dataset.id})")

Dataset cloned successfully: 'Meeting Transcripts' (ID: 13f2957b-abc5-4b0c-9200-e1f90829afe8)


### Experiments

Now, let's run some experiments on this dataset using two different prompts. Let's add an evaluator that tries to score how good our summaries are!

In [3]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain_openai import ChatOpenAI

# Standard client for the candidate summarizers
openai_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

SUMMARIZATION_SYSTEM_PROMPT = """You are a judge, aiming to score how well a summary summarizes the content of a transcript"""

SUMMARIZATION_HUMAN_PROMPT = """
[The Meeting Transcript] {transcript}
[The Start of Summarization] {summary} [The End of Summarization]"""

# Rule #10: Lowercase title contract
class SummarizationScore(BaseModel):
    model_config = {"title": "summarization_score"}
    score: int = Field(
        description="A score from 1-5 ranking how good the summarization is for the provided transcript, with 1 being a bad summary, and 5 being a great summary"
    )

# Structured Output Judge via Groq ($0)
score_judge = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0
).with_structured_output(SummarizationScore)


def summary_score_evaluator(inputs: dict, outputs: dict) -> dict:
    messages = [
        {"role": "system", "content": SUMMARIZATION_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": SUMMARIZATION_HUMAN_PROMPT.format(
                transcript=inputs.get("transcript", "")[:3000],  # Token Diet protection
                summary=outputs.get("output", "N/A"),
            )
        }
    ]
    result: SummarizationScore = score_judge.invoke(messages)
    return {"key": "summary_score", "score": result.score}

First, we'll run our experiment with a good version of our prompt!

In [4]:
# Candidate 1: 3-Sentence Comprehensive Prompt
def good_summarizer(inputs: dict):
    response = openai_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": f"Concisely summarize this meeting in 3 sentences. Make sure to include all of the important events. Meeting: {inputs['transcript'][:3000]}"
            }
        ],
        max_tokens=250,
    )
    return response.choices[0].message.content

print("Running Good Summarizer Experiment...")
good_exp = client.evaluate(
    good_summarizer,
    data=dataset,
    evaluators=[summary_score_evaluator],
    experiment_prefix="Good Summarizer",
    max_concurrency=2
)

Running Good Summarizer Experiment...
View the evaluation results for experiment: 'Good Summarizer-793e60f0' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/13f2957b-abc5-4b0c-9200-e1f90829afe8/compare?selectedSessions=63038e25-a370-4f31-8777-509d8a082dcd




Now, we'll run an experiment with a worse version of our prompt, to highlight the difference.

In [5]:
# Candidate 2: 1-Sentence Lazy Prompt
def bad_summarizer(inputs: dict):
    response = openai_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "user",
                "content": f"Summarize this in one sentence. {inputs['transcript'][:3000]}"
            }
        ],
        max_tokens=100,
    )
    return response.choices[0].message.content

print("Running Bad Summarizer Experiment...")
bad_exp = client.evaluate(
    bad_summarizer,
    data=dataset,
    evaluators=[summary_score_evaluator],
    experiment_prefix="Bad Summarizer",
    max_concurrency=2
)

Running Bad Summarizer Experiment...
View the evaluation results for experiment: 'Bad Summarizer-77a46920' at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/13f2957b-abc5-4b0c-9200-e1f90829afe8/compare?selectedSessions=b778922f-92f3-4dec-8752-f570adf5c74c




### Pairwise Experiment

Let's define a function that will compare our two experiments. These are the fields that pairwise evaluator functions get access to:
- `inputs: dict`: A dictionary of the inputs corresponding to a single example in a dataset.
- `outputs: list[dict]`: A list of the dict outputs produced by each experiment on the given inputs.
- `reference_outputs: dict`: A dictionary of the reference outputs associated with the example, if available.
- `runs: list[Run]`: A list of the full Run objects generated by the experiments on the given example. Use this if you need access to intermediate steps or metadata about each run.
- `example: Example`: The full dataset Example, including the example inputs, outputs (if available), and metdata (if available).

First, let's give our LLM-as-Judge some instructions. In our case, we're just going to directly use LLM-as-judge to grade which of the summarizers is the most helpful.

It might be hard to grade our summarizers without a ground truth reference, but here, comparing different prompts head to head will give us a sense of which is better!

In [6]:
JUDGE_SYSTEM_PROMPT = """
Please act as an impartial judge and evaluate the quality of the summarizations provided by two AI summarizers to the meeting transcript below.
Your evaluation should consider factors such as the helpfulness, relevance, accuracy, depth, creativity, and level of detail of their summarizations. 
Begin your evaluation by comparing the two summarizations and provide a short explanation. 
Avoid any position biases and ensure that the order in which the responses were presented does not influence your decision. 
Do not favor certain names of the assistants. 
Be as objective as possible. """

JUDGE_HUMAN_PROMPT = """
[The Meeting Transcript] {transcript}

[The Start of Assistant A's Summarization] {answer_a} [The End of Assistant A's Summarization]

[The Start of Assistant B's Summarization] {answer_b} [The End of Assistant B's Summarization]"""

Our function will take in an `inputs` dictionary, and a list of `outputs` dictionaries for the different experiments that we want to compare.

In [9]:
from pydantic import BaseModel, Field

class Preference(BaseModel):
    model_config = {"title": "preference"}
    preference: int = Field(
        description="1 if Assistant A answer is better. 2 if Assistant B answer is better. 0 if it is a tie."
    )

pref_judge = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0
).with_structured_output(Preference)


def ranked_preference(inputs: dict, outputs: list[dict]) -> list:
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": JUDGE_HUMAN_PROMPT.format(
                transcript=inputs.get("transcript", "")[:3000],
                answer_a=outputs[0].get("output", "N/A"),
                answer_b=outputs[1].get("output", "N/A")
            )
        }
    ]

    result: Preference = pref_judge.invoke(messages)
    preference_score = result.preference

    if preference_score == 1:
        return [1, 0]  # Assistant A wins
    elif preference_score == 2:
        return [0, 1]  # Assistant B wins
    else:
        return [0, 0]  # Tie

Now let's run our pairwise experiment with `evaluate()`

In [8]:
from langsmith import evaluate

# Automatically grabs the exact experiment names generated by LangSmith
exp_names = (good_exp.experiment_name, bad_exp.experiment_name)
print(f"Pitting {exp_names[0]} vs {exp_names[1]}...")

evaluate(
    exp_names,
    evaluators=[ranked_preference],
    max_concurrency=2
)

Pitting Good Summarizer-793e60f0 vs Bad Summarizer-77a46920...
View the pairwise evaluation results at:
https://smith.langchain.com/o/81d0633c-0684-4f49-9565-e453eca174ad/datasets/13f2957b-abc5-4b0c-9200-e1f90829afe8/compare?selectedSessions=63038e25-a370-4f31-8777-509d8a082dcd%2Cb778922f-92f3-4dec-8752-f570adf5c74c&comparativeExperiment=fe91a9e1-e791-4dbf-bb6d-f3ef8c047a96


